# Fink/LSST — Dipole Analysis: Reload from disk and reproduce figures

## Strategy

This notebook is the **fully-offline reload** variant of `03b_dipoleobjectcorr.ipynb`.

All data are read from the parquet files already written by `03b_dipoleobjectcorr.ipynb`
and stored in `data_DIPOLES_03b/`.  **No API call is made anywhere in this notebook.**

### Files read

| File | Content |
|------|---------|
| `data_DIPOLES_03b/presel_catalogue.parquet` | Pre-selected diaObjects (nDiaSources >= threshold) |
| `data_DIPOLES_03b/dipole_stats_from_sources.parquet` | Per-object dipole statistics |
| `data_DIPOLES_03b/all_src_presel.parquet` | All downloaded diaSources (all objects concatenated) |
| `data_DIPOLES_03b/topranked_objects_dipoles.parquet` | Top-ranked objects by dipole count |
| `data_DIPOLES_03b/dipole_angle_stability.csv` | Angular stability table |
| `data_DIPOLES_03b/src_per_object/{oid}_src.parquet` | Per-object diaSources |

### Figures reproduced (same as `03b`)

1. nDiaSources distribution histogram  
2. Stacked dipole count per object (top 50)  
3. Dipole count distribution + Lorenz curve + Gini coefficient  
4. n_src vs n_dipoles scatter plot  
5. psfFlux − apFlux Δm histograms (global + per band)  
6. Three-panel light curve for each top object  
7. Angular stability rose histogram  
8. Per-field stacked histogram  

Output figures go to `figs_DIPOLES_04/`.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-28

## 1. Imports & configuration

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── Input: data written by 03b_dipoleobjectcorr.ipynb ─────────────────────────
DIR_DATA_IN = "data_DIPOLES_03b"
DIR_SRC_PER_OBJ = os.path.join(DIR_DATA_IN, "src_per_object")

# ── Output figures directory (separate from 03b) ──────────────────────────────
NB_TAG = "DIPOLES_04"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── DDF definitions ───────────────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Pre-selection threshold (must match 03b) ──────────────────────────────────
NDIASOURCES_MIN = 500

# ── Light curve plotting ──────────────────────────────────────────────────────
TOP_N_OBJECTS = 10  # max objects in detailed light curve plots

# ── Zero-point for flux → magnitude conversion (AB system, nJy) ──────────────
FLUX0_NJY = 3.631e9  # 1 Jy = 1e9 nJy

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Configuration done.  NDIASOURCES_MIN={NDIASOURCES_MIN}")

## 2. Load catalogues from `data_DIPOLES_03b`

All files were written by `03b_dipoleobjectcorr.ipynb`.  No API call is needed.

In [ ]:
# ── Pre-selection catalogue ───────────────────────────────────────────────────
path_presel = os.path.join(DIR_DATA_IN, "presel_catalogue.parquet")
df_presel = pd.read_parquet(path_presel)
print(f"Pre-selection catalogue: {len(df_presel)} objects  ({path_presel})")
display(df_presel.head(10))

In [ ]:
# ── Dipole statistics ─────────────────────────────────────────────────────────
path_stats = os.path.join(DIR_DATA_IN, "dipole_stats_from_sources.parquet")
df_stats = pd.read_parquet(path_stats)
print(f"Dipole statistics: {len(df_stats)} objects  ({path_stats})")
print(f"  with >= 1 dipole  : {(df_stats['n_dipoles'] >= 1).sum()}")
print(f"  with >= 5 dipoles : {(df_stats['n_dipoles'] >= 5).sum()}")
print(f"  with >= 20 dipoles: {(df_stats['n_dipoles'] >= 20).sum()}")
display(df_stats.head(20))

In [ ]:
# ── All diaSources concatenated ───────────────────────────────────────────────
path_all_src = os.path.join(DIR_DATA_IN, "all_src_presel.parquet")
df_all_src = pd.read_parquet(path_all_src)

# Ensure boolean column is properly typed
if "r:isDipole" in df_all_src.columns:
    df_all_src["r:isDipole"] = df_all_src["r:isDipole"].fillna(False).astype(bool)
if "is_dipole" not in df_all_src.columns:
    df_all_src["is_dipole"] = (
        df_all_src.get("r:isDipole", pd.Series(False, index=df_all_src.index)).fillna(False).astype(bool)
    )

print(f"All diaSources: {len(df_all_src):,} rows  ({path_all_src})")
print(f"  dipole-flagged: {df_all_src['is_dipole'].sum():,}")
print(f"  non-dipole    : {(~df_all_src['is_dipole']).sum():,}")
print(f"  columns: {list(df_all_src.columns)}")

In [ ]:
# ── Top-ranked objects ────────────────────────────────────────────────────────
path_top = os.path.join(DIR_DATA_IN, "topranked_objects_dipoles.parquet")
df_top_ranked = pd.read_parquet(path_top)
print(f"Top-ranked objects: {len(df_top_ranked)} rows")
display(df_top_ranked)

In [ ]:
# ── Angular stability table ───────────────────────────────────────────────────
path_angles = os.path.join(DIR_DATA_IN, "dipole_angle_stability.csv")
df_angles = pd.read_csv(path_angles)
print(f"Angular stability table: {len(df_angles)} objects")
display(df_angles)

In [ ]:
# ── Per-object diaSources: build src_cache dict ───────────────────────────────
# Keys are diaObjectId strings, values are DataFrames.
src_cache: dict[str, pd.DataFrame] = {}
pq_files = sorted(glob.glob(os.path.join(DIR_SRC_PER_OBJ, "*_src.parquet")))
print(f"Loading {len(pq_files)} per-object parquet files from {DIR_SRC_PER_OBJ} …")
for fpath in pq_files:
    oid = os.path.basename(fpath).replace("_src.parquet", "")
    df = pd.read_parquet(fpath)
    # Ensure boolean dipole column
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)
    if "is_dipole" not in df.columns:
        df["is_dipole"] = df.get("r:isDipole", pd.Series(False, index=df.index)).fillna(False).astype(bool)
    # Ensure numeric types for key columns
    for col in (
        "r:midpointMjdTai",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleFluxDiff",
        "r:dipoleMeanFlux",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    src_cache[oid] = df
n_ok = sum(1 for v in src_cache.values() if not v.empty)
print(f"Loaded {n_ok}/{len(pq_files)} objects with data.")

In [ ]:
# ── Build presel dict (needed by plotting functions) ──────────────────────────
# Reconstruct the lightweight presel dict from the parquet catalogue.
presel: dict[str, dict] = {}
for _, row in df_presel.iterrows():
    oid = str(row["diaObjectId"])
    presel[oid] = {
        "nDiaSources": int(row["nDiaSources"]),
        "ra": float(row.get("ra", np.nan)),
        "dec": float(row.get("dec", np.nan)),
        "field": str(row.get("field", "unknown")),
        "gaia_name": row.get("gaia_name"),
        "simbad": row.get("simbad"),
        "label": row.get("label"),
        "label_clf": row.get("label_clf"),
    }
print(f"presel dict: {len(presel)} objects")

## 3. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """Convert MJD (TAI) → list of 'YYYY-MM-DD' strings."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 8) -> None:
    """Add a secondary x-axis on top of *ax* showing calendar dates (YYYY-MM-DD)."""
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    tick_mjd = np.linspace(mjd_lo, mjd_hi, max(3, min(n_ticks, len(finite))))
    tick_lbls = mjd_to_datestr(tick_mjd)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


def circular_std_deg(angles_deg: np.ndarray) -> float:
    """Circular std of angles (degrees), folded mod 180° (dipole symmetry)."""
    a = np.deg2rad(np.asarray(angles_deg, dtype=float) % 180)
    R = np.abs(np.mean(np.exp(2j * a)))
    return float(np.rad2deg(np.sqrt(-2 * np.log(R + 1e-12))) / 2)


print("Utility functions defined.")

## 4. nDiaSources distribution

In [ ]:
nv = df_presel["nDiaSources"].values
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(nv, bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[0].axvline(NDIASOURCES_MIN, color="tomato", lw=1.5, ls="--", label=f"threshold = {NDIASOURCES_MIN}")
axes[0].set_xlabel("nDiaSources")
axes[0].set_ylabel("N objects")
axes[0].set_title(f"nDiaSources distribution (all >= {NDIASOURCES_MIN})")
axes[0].legend(fontsize=8)

bins_log = np.logspace(np.log10(max(nv.min(), 1)), np.log10(nv.max() + 1), 40)
axes[1].hist(nv, bins=bins_log, color="steelblue", edgecolor="white", lw=0.3)
axes[1].axvline(NDIASOURCES_MIN, color="tomato", lw=1.5, ls="--")
axes[1].set_xscale("log")
axes[1].set_xlabel("nDiaSources")
axes[1].set_title("nDiaSources distribution (log)")

plt.tight_layout()
savefig(f"nDiaSources_distribution_min{NDIASOURCES_MIN}")
plt.show()

## 5. Dipole statistics summary

In [ ]:
print(f"Statistics for {len(df_stats)} objects:")
print(f"  with >= 1 dipole  : {(df_stats['n_dipoles'] >= 1).sum()}")
print(f"  with >= 5 dipoles : {(df_stats['n_dipoles'] >= 5).sum()}")
print(f"  with >= 20 dipoles: {(df_stats['n_dipoles'] >= 20).sum()}")
display(df_stats.head(20))

## 6. Stacked histogram: dipole count per object, stacked by band

Each bar = one `diaObjectId`.  Stacked colours = per-band dipole counts.  
Objects sorted by total dipole count descending.

In [ ]:
df_dip_nz = df_stats[df_stats["n_dipoles"] > 0].copy()
df_dip_nz["rank"] = df_dip_nz["n_dipoles"].rank(method="dense", ascending=False).astype(int)

if df_dip_nz.empty:
    print("No dipoles found in the loaded data.")
else:
    N_SHOW = min(50, len(df_dip_nz))
    top_df = df_dip_nz.head(N_SHOW)
    x_pos = np.arange(N_SHOW)
    bottom = np.zeros(N_SHOW)

    fig, ax = plt.subplots(figsize=(max(12, N_SHOW * 0.22), 5))
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_df.columns:
            continue
        vals = top_df[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
            width=0.85,
        )
        bottom += vals
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(oid) for oid in top_df["diaObjectId"]], rotation=90, fontsize=6)
    ax.set_xlabel("diaObjectId")
    ax.set_ylabel("Number of dipole detections")
    ax.set_title(
        f"Dipole count per diaObject — top {N_SHOW} (stacked by band)\n"
        f"Pre-selection: nDiaSources >= {NDIASOURCES_MIN}  |  "
        f"Objects with >=1 dipole: {len(df_dip_nz)}"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_stacked_per_object_min{NDIASOURCES_MIN}")
    plt.show()

display(df_dip_nz.head(20))

## 7. Dipole count distribution + Lorenz curve + Gini coefficient

The **Lorenz curve** quantifies concentration of dipoles across diaObjects.  
$G = 1 - 2\int_0^1 L(x)\,dx$; $G=0$ = uniform, $G=1$ = one object monopolises all dipoles.

In [ ]:
if not df_dip_nz.empty:
    counts = df_dip_nz["n_dipoles"].values
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # --- Linear histogram ---
    axes[0].hist(counts, bins=40, color="steelblue", edgecolor="white", lw=0.3)
    axes[0].set_xlabel("n_dipoles per object")
    axes[0].set_ylabel("N objects")
    axes[0].set_title(f"Dipole count distribution (linear)  [nDiaSrc>={NDIASOURCES_MIN}]")

    # --- Log-log histogram ---
    bins_log = np.logspace(0, np.log10(counts.max() + 1), 30)
    axes[1].hist(counts, bins=bins_log, color="tomato", edgecolor="white", lw=0.3)
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("n_dipoles per object")
    axes[1].set_title("Dipole count distribution (log-log)")

    # --- Lorenz curve ---
    sc = np.sort(counts)  # ascending: poorest first
    cum = np.cumsum(sc) / sc.sum()
    obj = np.arange(1, len(sc) + 1) / len(sc)

    axes[2].plot(obj * 100, cum * 100, color="steelblue", lw=2, label="Lorenz curve")
    axes[2].fill_between(obj * 100, obj * 100, cum * 100, alpha=0.15, color="steelblue", label="Gini area")
    axes[2].plot([0, 100], [0, 100], "--", color="grey", lw=1, label="equal distribution")

    # Annotate top 10 %
    idx10 = max(1, int(0.10 * len(sc)))
    frac10 = (1 - cum[-(idx10)]) * 100
    axes[2].axvline(90, color="tomato", lw=1, ls=":")
    axes[2].text(
        74, frac10 / 2 + 5, f"top 10% objects\n→ {frac10:.0f}% of dipoles", color="tomato", fontsize=8
    )

    # Gini coefficient
    gini = 1.0 - 2.0 * float(np.trapz(cum, obj))
    axes[2].text(
        75,
        88,
        f"Gini = {gini:.3f}",
        fontsize=10,
        color="steelblue",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="steelblue", alpha=0.8),
    )

    axes[2].set_xlabel("Cumulative fraction of objects (%, poorest first)")
    axes[2].set_ylabel("Cumulative fraction of dipoles (%)")
    axes[2].set_title("Lorenz curve — dipole concentration")
    axes[2].set_xlim(0, 100)
    axes[2].set_ylim(0, 100)
    axes[2].legend(fontsize=8, loc="upper left")

    plt.tight_layout()
    savefig(f"dipole_distribution_lorenz_min{NDIASOURCES_MIN}")
    plt.show()

    print(f"\nGini coefficient = {gini:.4f}")
    if gini > 0.6:
        print("  → G > 0.6: very unequal — dipoles dominated by a small number of objects.")
    elif gini > 0.3:
        print("  → 0.3 < G ≤ 0.6: moderately unequal distribution.")
    else:
        print("  → G ≤ 0.3: relatively uniform distribution.")

## 8. Select top high-dipole objects — n_src vs n_dipoles scatter

In [ ]:
top_sel = df_stats[df_stats["n_dipoles"] > 0].head(TOP_N_OBJECTS).copy()
print(f"Top {TOP_N_OBJECTS} objects by dipole count:")
cols_show = [
    c
    for c in [
        "diaObjectId",
        "field",
        "nDiaSources",
        "n_src",
        "n_dipoles",
        "dipole_fraction",
        "gaia_name",
        "simbad",
        "label",
    ]
    if c in top_sel.columns
]
display(top_sel[cols_show])

fig, ax = plt.subplots(figsize=(7, 5))
for fld in DEEP_FIELDS:
    sub = df_stats[df_stats["field"] == fld]
    ax.scatter(sub["n_src"], sub["n_dipoles"], s=12, alpha=0.5, label=fld)
ax.scatter(
    top_sel["n_src"].values,
    top_sel["n_dipoles"].values,
    s=90,
    marker="*",
    color="gold",
    edgecolors="k",
    lw=0.5,
    zorder=5,
    label=f"top {TOP_N_OBJECTS}",
)
ax.set_xlabel("n_src (downloaded diaSources)")
ax.set_ylabel("n_dipoles")
ax.set_title(f"Sources vs dipoles per diaObject  [nDiaSrc>={NDIASOURCES_MIN}]")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
savefig(f"nsrc_vs_ndipoles_scatter_min{NDIASOURCES_MIN}")
plt.show()

## 9. psfFlux − apFlux magnitude difference: dipole vs non-dipole

$$\Delta m = m_{\rm psf} - m_{\rm ap} = -2.5 \log_{10}\!\left(\frac{F_{\rm psf}}{F_{\rm ap}}\right)$$

Read from the concatenated `all_src_presel.parquet`.  No API call.

In [ ]:
has_ap = "r:apFlux" in df_all_src.columns and df_all_src["r:apFlux"].notna().any()

if not has_ap:
    print("r:apFlux not available in all_src_presel.parquet — skipping Δm section.")
else:
    psf = pd.to_numeric(df_all_src["r:psfFlux"], errors="coerce")
    ap = pd.to_numeric(df_all_src["r:apFlux"], errors="coerce")
    with np.errstate(invalid="ignore", divide="ignore"):
        delta_m = np.where(
            (psf.values > 0) & (ap.values > 0),
            -2.5 * np.log10(psf.values / ap.values),
            np.nan,
        )
    df_all_src["delta_m_psf_ap"] = delta_m

    n_total = df_all_src["delta_m_psf_ap"].notna().sum()
    is_dip = df_all_src["is_dipole"].values
    print(f"delta_m_psf_ap computed for {n_total:,} sources")
    print(f"  dipole  median Δm = {np.nanmedian(delta_m[is_dip]):.4f} mag")
    print(f"  non-dip median Δm = {np.nanmedian(delta_m[~is_dip]):.4f} mag")

In [ ]:
# ── Global histogram (all bands combined) ─────────────────────────────────────
if has_ap and "delta_m_psf_ap" in df_all_src.columns:
    CLIP = 1.0
    BINS = np.linspace(-CLIP, CLIP, 81)

    fig, ax = plt.subplots(figsize=(7, 4))
    dm_nd = df_all_src.loc[~df_all_src["is_dipole"], "delta_m_psf_ap"].dropna().values
    dm_dip = df_all_src.loc[df_all_src["is_dipole"], "delta_m_psf_ap"].dropna().values

    ax.hist(
        np.clip(dm_nd, -CLIP, CLIP),
        bins=BINS,
        density=True,
        alpha=0.55,
        color="steelblue",
        label=f"non-dipole  (N={len(dm_nd):,})",
    )
    ax.hist(
        np.clip(dm_dip, -CLIP, CLIP),
        bins=BINS,
        density=True,
        alpha=0.70,
        color="tomato",
        label=f"dipole  (N={len(dm_dip):,})",
    )

    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.5)
    ax.axvline(
        float(np.nanmedian(dm_nd)),
        color="steelblue",
        lw=1.5,
        ls=":",
        label=f"non-dipole median = {np.nanmedian(dm_nd):.3f}",
    )
    ax.axvline(
        float(np.nanmedian(dm_dip)),
        color="tomato",
        lw=1.5,
        ls=":",
        label=f"dipole median = {np.nanmedian(dm_dip):.3f}",
    )

    ax.set_xlabel(r"$\Delta m = m_{\rm psf} - m_{\rm ap}$ (mag)")
    ax.set_ylabel("Probability density")
    ax.set_title(
        r"$m_{\rm psf} - m_{\rm ap}$ distribution — all bands combined" + "\n"
        f"Pre-selection: nDiaSrc >= {NDIASOURCES_MIN}"
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig(f"delta_mag_psf_ap_all_bands_min{NDIASOURCES_MIN}")
    plt.show()

In [ ]:
# ── Per-band histograms ────────────────────────────────────────────────────────
if has_ap and "delta_m_psf_ap" in df_all_src.columns and "r:band" in df_all_src.columns:
    bands_present = [b for b in BAND_ORDER if b in df_all_src["r:band"].values]
    ncols_b = min(3, len(bands_present))
    nrows_b = int(np.ceil(len(bands_present) / ncols_b))
    fig, axes = plt.subplots(nrows_b, ncols_b, figsize=(5 * ncols_b, 3.5 * nrows_b), squeeze=False)

    for bidx, band in enumerate(bands_present):
        ax = axes[bidx // ncols_b][bidx % ncols_b]
        sub = df_all_src[df_all_src["r:band"] == band]
        dm_nd_b = sub.loc[~sub["is_dipole"], "delta_m_psf_ap"].dropna().values
        dm_dip_b = sub.loc[sub["is_dipole"], "delta_m_psf_ap"].dropna().values
        color = BAND_COLORS[band]

        if len(dm_nd_b) > 0:
            ax.hist(
                np.clip(dm_nd_b, -CLIP, CLIP),
                bins=BINS,
                density=True,
                alpha=0.45,
                color="steelblue",
                label=f"non-dipole (N={len(dm_nd_b):,})",
            )
            ax.axvline(float(np.nanmedian(dm_nd_b)), color="steelblue", lw=1.5, ls=":")
        if len(dm_dip_b) > 0:
            ax.hist(
                np.clip(dm_dip_b, -CLIP, CLIP),
                bins=BINS,
                density=True,
                alpha=0.70,
                color=color,
                label=f"dipole (N={len(dm_dip_b):,})",
            )
            ax.axvline(float(np.nanmedian(dm_dip_b)), color=color, lw=1.5, ls=":")
        else:
            ax.text(0.5, 0.5, "no dipoles", ha="center", va="center", transform=ax.transAxes, fontsize=9)
        ax.axvline(0, color="k", lw=0.8, ls="--", alpha=0.4)
        ax.set_xlabel(r"$\Delta m$ (mag)")
        ax.set_ylabel("Density")
        ax.set_title(f"Band {band}", fontsize=9)
        ax.legend(fontsize=7)

    for bidx in range(len(bands_present), nrows_b * ncols_b):
        axes[bidx // ncols_b][bidx % ncols_b].set_visible(False)

    fig.suptitle(
        r"$m_{\rm psf} - m_{\rm ap}$ per band — dipole vs non-dipole" + f"\n[nDiaSrc>={NDIASOURCES_MIN}]",
        fontsize=11,
        y=1.01,
    )
    plt.tight_layout()
    savefig(f"delta_mag_psf_ap_per_band_min{NDIASOURCES_MIN}")
    plt.show()

## 10. Three-panel light curve for top high-dipole objects

**Panel 1** — psfFlux light curve (per band, dipoles circled).  
**Panel 2** — nightly dipole histogram (stacked by band, cumulative line).  
**Panel 3** — dipole morphology: `dipoleLength` (left y) and `dipoleAngle` mod 360° (right y).

In [ ]:
def plot_object_lc(
    oid: str,
    df_src: pd.DataFrame,
    meta: dict,
    stat: dict,
    flux_col: str = "r:psfFlux",
    ferr_col: str = "r:psfFluxErr",
) -> None:
    """Three-panel light curve + nightly dipole histogram + dipole morphology."""
    if df_src.empty:
        print(f"  {oid}: empty diaSources — skipping.")
        return

    df = df_src.sort_values("r:midpointMjdTai").copy()
    df["is_dipole"] = (
        df.get("is_dipole", df.get("r:isDipole", pd.Series(False, index=df.index))).fillna(False).astype(bool)
    )
    mjd_all = pd.to_numeric(df["r:midpointMjdTai"], errors="coerce").values

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), gridspec_kw={"height_ratios": [3, 1.5, 1.5]})

    # ── Panel 1 : psfFlux light curve ─────────────────────────────────────────
    ax1 = axes[0]
    for band in BAND_ORDER:
        sub = df[df["r:band"] == band] if "r:band" in df.columns else pd.DataFrame()
        if sub.empty:
            continue
        flux = pd.to_numeric(sub[flux_col], errors="coerce").values
        ferr = pd.to_numeric(sub[ferr_col], errors="coerce").values if ferr_col in sub.columns else None
        mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
        color = BAND_COLORS[band]
        ax1.errorbar(
            mjd_b,
            flux,
            yerr=ferr,
            fmt="o",
            ms=4,
            lw=0.8,
            capsize=2,
            capthick=0.8,
            color=color,
            ecolor=color,
            alpha=0.8,
            label=f"{band} (n={len(sub)})",
        )
        dip = sub[sub["is_dipole"]]
        if not dip.empty:
            flux_d = pd.to_numeric(dip[flux_col], errors="coerce").values
            mjd_d = pd.to_numeric(dip["r:midpointMjdTai"], errors="coerce").values
            ax1.scatter(
                mjd_d,
                flux_d,
                s=130,
                facecolors="none",
                edgecolors="grey",
                linewidths=1.8,
                zorder=5,
                label=f"dipole {band} (n={len(dip)})",
            )

    ax1.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
    ax1.set_ylabel(f"{flux_col.split(':')[1]} (nJy)")
    ax1.legend(loc="best", fontsize=7, ncol=4)
    title = (
        f"diaObjectId={oid}  field={meta.get('field', '?')}  "
        f"nDiaSrc={meta.get('nDiaSources', '?')}  downloaded={stat.get('n_src', '?')}  "
        f"n_dip={stat.get('n_dipoles', '?')}  frac={stat.get('dipole_fraction', 0.0) * 100:.1f}%"
    )
    if meta.get("gaia_name") and str(meta["gaia_name"]) not in ("nan", "None", ""):
        title += f"  Gaia={meta['gaia_name']}"
    if meta.get("simbad") and str(meta["simbad"]) not in ("nan", "None", ""):
        title += f"  SIMBAD={meta['simbad']}"
    ax1.set_title(title, fontsize=8)
    add_date_axis_on_top(ax1, mjd_all, n_ticks=8)

    # ── Panel 2 : nightly dipole histogram ────────────────────────────────────
    ax2 = axes[1]
    df_dip = df[df["is_dipole"]].copy()
    if not df_dip.empty and "r:band" in df_dip.columns:
        df_dip["night"] = np.floor(pd.to_numeric(df_dip["r:midpointMjdTai"], errors="coerce").values).astype(
            int
        )
        night_band = (
            df_dip.groupby(["night", "r:band"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=BAND_ORDER, fill_value=0)
        )
        night_band["total"] = night_band.sum(axis=1)
        nights_mjd = night_band.index.values.astype(float) + 0.5
        bottom2 = np.zeros(len(night_band))
        for band in BAND_ORDER:
            if band not in night_band.columns:
                continue
            vals = night_band[band].values.astype(float)
            ax2.bar(
                nights_mjd,
                vals,
                bottom=bottom2,
                width=0.8,
                color=BAND_COLORS[band],
                edgecolor="white",
                lw=0.3,
                label=f"band {band}",
            )
            bottom2 += vals
        cum = np.cumsum(night_band["total"].values)
        ax2r = ax2.twinx()
        ax2r.step(nights_mjd, cum, where="post", color="k", lw=1.5, ls="--", alpha=0.6, label="cumulative")
        ax2r.set_ylabel("Cumulative dipoles", fontsize=8)
        ax2r.tick_params(axis="y", labelsize=8)
    ax2.set_ylabel("N dipoles per night")
    ax2.set_xlabel("MJD (TAI)")
    ax2.legend(loc="upper left", fontsize=7, ncol=3)

    finite_mjd = mjd_all[np.isfinite(mjd_all)]
    if len(finite_mjd) > 1:
        xlim = (finite_mjd.min() - 1, finite_mjd.max() + 1)
        ax1.set_xlim(xlim)
        ax2.set_xlim(xlim)

    # ── Panel 3 : dipole morphology ───────────────────────────────────────────
    ax3 = axes[2]
    if not df_dip.empty:
        for band in BAND_ORDER:
            sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
            if sub.empty or "r:dipoleLength" not in sub.columns:
                continue
            dl = pd.to_numeric(sub["r:dipoleLength"], errors="coerce").values
            mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
            ax3.scatter(mjd_b, dl, s=25, color=BAND_COLORS[band], marker="o", label=f"length {band}")
        if "r:dipoleAngle" in df_dip.columns:
            ax3r = ax3.twinx()
            for band in BAND_ORDER:
                sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
                if sub.empty:
                    continue
                da = pd.to_numeric(sub["r:dipoleAngle"], errors="coerce").values % 360
                mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
                ax3r.scatter(mjd_b, da, s=25, color=BAND_COLORS[band], marker="^", alpha=0.6)
            ax3r.set_ylabel("Dipole angle (deg)", fontsize=8, color="grey")
            ax3r.set_ylim(0, 360)
            ax3r.tick_params(axis="y", labelcolor="grey", labelsize=8)
        ax3.set_ylabel("Dipole length (arcsec)")
        ax3.set_xlabel("MJD (TAI)")
        ax3.legend(loc="upper left", fontsize=7, ncol=3)
        ax3.set_xlim(ax1.get_xlim())

    plt.tight_layout()
    savefig(f"lc_{oid}")
    plt.show()


print("plot_object_lc() defined.")

In [ ]:
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    if oid not in src_cache or src_cache[oid].empty:
        print(f"  {oid}: no per-object parquet found — skipping.")
        continue
    print(
        f"\n=== {oid}  field={srow.get('field', '?')}  "
        f"nDiaSrc={srow.get('nDiaSources', '?')}  n_dip={srow.get('n_dipoles', '?')} ==="
    )
    plot_object_lc(
        oid=oid,
        df_src=src_cache[oid],
        meta=presel.get(oid, {}),
        stat=srow.to_dict(),
    )
print("Done.")

## 11. Angular stability of dipole direction per object

The table was saved by `03b`; we just display it and add the rose histogram.

In [ ]:
print("Angular stability of dipole direction (sorted by circular std):")
display(df_angles)

In [ ]:
# ── Rose histogram of dipole angles — top objects combined ────────────────────
all_angles, all_bands = [], []
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    df = src_cache.get(oid, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns or "r:dipoleAngle" not in df.columns:
        continue
    df_dip = df[df.get("is_dipole", df["r:isDipole"].fillna(False).astype(bool))]
    ang = pd.to_numeric(df_dip["r:dipoleAngle"], errors="coerce").dropna().values
    bnd = df_dip["r:band"].values[: len(ang)] if "r:band" in df_dip.columns else ["?"] * len(ang)
    all_angles.extend(ang % 180)
    all_bands.extend(bnd)

if all_angles:
    n_bins = 36
    bins_a = np.linspace(0, 180, n_bins + 1)
    bottom3 = np.zeros(n_bins)
    fig, ax = plt.subplots(figsize=(7, 4))
    for band in BAND_ORDER:
        ang_b = np.array([a for a, b in zip(all_angles, all_bands) if b == band])
        if len(ang_b) == 0:
            continue
        cnt, _ = np.histogram(ang_b, bins=bins_a)
        ax.bar(
            (bins_a[:-1] + bins_a[1:]) / 2,
            cnt,
            bottom=bottom3,
            width=180 / n_bins,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
        )
        bottom3 += cnt
    ax.set_xlabel("Dipole angle mod 180° (degrees)")
    ax.set_ylabel("N detections")
    ax.set_xlim(0, 180)
    ax.set_title(f"Dipole angle distribution — top {TOP_N_OBJECTS} objects\n(folded mod 180° for ±symmetry)")
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_angle_histogram_top{TOP_N_OBJECTS}")
    plt.show()
else:
    print("No dipole angles available.")

## 12. Per-field stacked histogram

In [ ]:
n_fields = len(DEEP_FIELDS)
ncols = min(3, n_fields)
nrows = int(np.ceil(n_fields / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for idx, fld in enumerate(DEEP_FIELDS):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_stats[(df_stats["field"] == fld) & (df_stats["n_dipoles"] > 0)]
    if sub.empty:
        ax.set_title(f"{fld} — no dipoles")
        continue
    sub = sub.sort_values("n_dipoles", ascending=False)
    N_F = min(40, len(sub))
    top_f = sub.head(N_F)
    x_pos = np.arange(N_F)
    bottom = np.zeros(N_F)
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_f.columns:
            continue
        vals = top_f[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.2,
            label=band,
            width=0.85,
        )
        bottom += vals
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(o) for o in top_f["diaObjectId"]], rotation=90, fontsize=5)
    ax.set_title(f"{fld}  ({len(sub)} with >0 dip, top {N_F})", fontsize=8)
    ax.set_ylabel("N dipoles")
    ax.legend(loc="upper right", fontsize=6, ncol=3)

for idx in range(n_fields, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(f"Dipole count per diaObject — per DDF  [nDiaSrc>={NDIASOURCES_MIN}]", fontsize=11, y=1.01)
plt.tight_layout()
savefig(f"dipole_per_ddf_min{NDIASOURCES_MIN}")
plt.show()

## 13. Summary

| Step | Source |
|------|--------|
| Pre-selection catalogue | `data_DIPOLES_03b/presel_catalogue.parquet` |
| Per-object dipole stats | `data_DIPOLES_03b/dipole_stats_from_sources.parquet` |
| All diaSources | `data_DIPOLES_03b/all_src_presel.parquet` |
| Per-object sources | `data_DIPOLES_03b/src_per_object/{oid}_src.parquet` |
| Angular stability | `data_DIPOLES_03b/dipole_angle_stability.csv` |
| **No API call made** | — |

All figures saved to `figs_DIPOLES_04/`.
